# Experiment 1.3c-i: Isolate `lambda_max` from `alpha`

## Counterpart identity and truthful scope

This notebook is the narrative counterpart to:

- `experiments/Tier_1_Core_Mechanism_Tests/Exp_1.3_Singular_Value_Spectrum_Evolution/1.3c_Spectrum_Connection_to_WeightWatcher_alpha/1.3c-i_Isolate_lambda_max_from_alpha/run_experiment.py`

It is intentionally designed to **import and use the script's exact `run_experiment()` engine** rather than duplicating the training code. The goal of this first completion pass is alignment and truthfulness:

- preserve the current toy single-run study,
- present the results in a more academically serious and self-explanatory way,
- keep the script and notebook on the same computational path.

### Scope of the evidence in this notebook

This is a **single-seed, per-layer `W^T W` spectrum proxy study** in a deep linear network. It compares SGD-with-momentum and Muon on the same fixed synthetic problem.

The reported `alpha` is **not** a full WeightWatcher fit. Here it is a simpler **rank-decay proxy** obtained by regressing `log(eigenvalue)` against `log(rank)` on each layer's `W^T W` eigenvalues.


In [ ]:
from pathlib import Path
import importlib.util
import time
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string())
        else:
            print(obj)


In [ ]:
TARGET_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/Exp_1.3_Singular_Value_Spectrum_Evolution/1.3c_Spectrum_Connection_to_WeightWatcher_alpha/1.3c-i_Isolate_lambda_max_from_alpha/run_experiment.py')

def locate_script_path():
    search_roots = [Path.cwd(), *Path.cwd().parents]

    for root in search_roots:
        candidate = root / TARGET_RELATIVE
        if candidate.exists():
            return candidate.resolve()

    for root in search_roots:
        candidate = root / 'run_experiment.py'
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        'Could not locate run_experiment.py from the current working directory. '
        'Try starting Jupyter from the repository root or from this experiment directory.'
    )

SCRIPT_PATH = locate_script_path()
EXPERIMENT_DIR = SCRIPT_PATH.parent

spec = importlib.util.spec_from_file_location('exp_1_3c_i_module', SCRIPT_PATH)
exp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp)

print('Notebook counterpart located successfully:')
print(f'  script path:     {SCRIPT_PATH}')
print(f'  experiment dir:  {EXPERIMENT_DIR}')


## What is actually measured here, and what is not

### Measured

For each optimizer, at every recorded step, the script returns per-layer measurements of:

- the full eigenvalue spectrum of each layer's `W^T W`,
- `lambda_max`, `lambda_median`, and `lambda_min`,
- `lambda_max / lambda_median` as a simple outlier-ratio diagnostic,
- `alpha` from the rank-decay log-log regression,
- `alpha_clipped`, defined by refitting after removing the top eigenvalue,
- training loss.

### Not measured

This notebook does **not** establish:

- multi-seed uncertainty,
- a calibrated WeightWatcher heavy-tail fit,
- product-matrix spectral statistics,
- test accuracy or generalization,
- a definitive optimizer-fairness study.

In particular, the SGD learning rate is chosen by a short candidate-grid stability heuristic in the script. That is a reasonable toy-comparison convenience, but it is not a formal fairness guarantee.


In [ ]:
notebook_wall_start = time.perf_counter()

experiment = exp.run_experiment()
artifact_paths = exp.save_results_bundle(experiment, output_dir=EXPERIMENT_DIR)
experiment['artifacts'] = artifact_paths

notebook_wall_seconds = time.perf_counter() - notebook_wall_start

print('Experiment executed via imported script engine.')
print(f"  compute runtime reported by script engine: {experiment['metadata']['runtime_seconds']:.3f} s")
print(f"  notebook wall-clock runtime (run + save): {notebook_wall_seconds:.3f} s")
print(f"  chosen SGD LR: {experiment['learning_rate_search']['chosen_sgd_lr']}")
print(f"  final SGD loss:  {experiment['optimizers']['SGD']['final_loss']:.6e}")
print(f"  final Muon loss: {experiment['optimizers']['Muon']['final_loss']:.6e}")


## Reproducibility, runtime, and configuration

The cells below log the identity of the run, the preserved default configuration, the fixed-seed setup, and the main runtime/optimizer outputs needed to reproduce the present notebook state.


In [ ]:
run_overview_df = pd.DataFrame([
    {'field': 'experiment_id', 'value': experiment['metadata']['experiment_id']},
    {'field': 'study_scope', 'value': experiment['metadata']['study_scope']},
    {'field': 'seed', 'value': experiment['seed']},
    {'field': 'script_path', 'value': experiment['metadata']['script_path']},
    {'field': 'counterpart_notebook', 'value': experiment['metadata']['counterpart_notebook']},
    {'field': 'compute_runtime_seconds', 'value': round(experiment['metadata']['runtime_seconds'], 6)},
    {'field': 'notebook_wall_seconds', 'value': round(notebook_wall_seconds, 6)},
    {'field': 'chosen_sgd_lr', 'value': experiment['learning_rate_search']['chosen_sgd_lr']},
    {'field': 'fixed_muon_lr', 'value': experiment['config']['lr_muon']},
    {'field': 'initial_loss', 'value': experiment['initial_loss']},
    {'field': 'final_sgd_loss', 'value': experiment['optimizers']['SGD']['final_loss']},
    {'field': 'final_muon_loss', 'value': experiment['optimizers']['Muon']['final_loss']},
    {'field': 'overall_verdict', 'value': experiment['summary']['overall']},
])

config_df = pd.DataFrame([
    {'config_key': key, 'value': value}
    for key, value in experiment['config'].items()
])

problem_stats_df = pd.DataFrame([
    {'problem_stat': key, 'value': value}
    for key, value in experiment['problem_stats'].items()
])

optimizer_df = pd.DataFrame([
    {
        'optimizer': name,
        'learning_rate': result['learning_rate'],
        'steps_completed': result['steps_completed'],
        'diverged': result['diverged'],
        'runtime_seconds': result['runtime_seconds'],
        'initial_loss': result['initial_loss'],
        'final_loss': result['final_loss'],
    }
    for name, result in experiment['optimizers'].items()
])

artifact_df = pd.DataFrame([
    {'artifact': key, 'path': value}
    for key, value in experiment['artifacts'].items()
])

print('Run overview')
display(run_overview_df)
print()
print('Configuration')
display(config_df)
print()
print('Fixed-problem diagnostics')
display(problem_stats_df)
print()
print('Optimizer-level runtime and loss summary')
display(optimizer_df)
print()
print('Saved artifacts')
display(artifact_df)


## Learning-rate selection note

The comparison keeps Muon's learning rate fixed and chooses SGD's learning rate by scanning a small candidate list for a non-divergent 200-step run. The table below is included so that the notebook is explicit about that heuristic rather than silently treating it as a perfectly fair or formal tuning procedure.


In [ ]:
lr_scan_df = pd.DataFrame(experiment['learning_rate_search']['scan'])
display(lr_scan_df)

print(textwrap.fill(
    'Interpretation: the script chooses the first candidate that remains non-divergent under a short stability screen. '
    'That preserves the original experiment design, but it should be described as a heuristic rather than a rigorous fairness guarantee.',
    width=110,
))


## What the `alpha` proxy is, and what it is not

In this pair, `alpha` should be interpreted narrowly:

- **What it is**: a deterministic, per-layer rank-decay summary of the `W^T W` eigenvalue spectrum.
- **How it is computed**: sort eigenvalues in descending order, regress `log(eigenvalue)` on `log(rank)`, and report the negative slope.
- **Why it is useful here**: it gives a simple way to compare whether the *bulk* of the layer spectrum changes differently under SGD and Muon.

But it is **not**:

- a full WeightWatcher empirical spectral density analysis,
- a calibrated statement about heavy-tail universality classes,
- a direct generalization metric.

Throughout this notebook, lower `alpha` means a **flatter rank-decay fit in this toy proxy**, not a canonical WeightWatcher regime label.


## Trajectories over training

The next figure is produced directly from the script's returned results. It shows the layer-mean trajectories for:

- `alpha`,
- `alpha_clipped`,
- `lambda_max / lambda_median`,
- `lambda_max`,
- clipping effect `|alpha - alpha_clipped|`,
- loss.


In [ ]:
fig, axes = exp.create_summary_figure(experiment, show=False, close=False, use_agg=False)
plt.show()
plt.close(fig)


### Trajectory interpretation

The most important qualitative pattern to watch is whether the optimizers separate on the *bulk-spectrum proxy* (`alpha`) as well as on the top-eigenvalue diagnostics (`lambda_max` and outlier ratio).

In the present run, the interesting outcome is not merely that Muon has a somewhat smaller `lambda_max`, but that the layer-mean `alpha` trajectory also moves differently. That is the main reason T1/T2/T3 can all be supported while T4 still fails.


In [ ]:
selected_steps = experiment['config']['table_steps']

trajectory_rows = []
for optimizer_name in ['SGD', 'Muon']:
    result = experiment['optimizers'][optimizer_name]
    summary = result['trajectory_summary']
    for step in selected_steps:
        idx = exp.step_idx(result['measure_steps'], step)
        if idx is None:
            continue
        trajectory_rows.append({
            'optimizer': optimizer_name,
            'step': step,
            'alpha_mean': summary['alpha_mean'][idx],
            'alpha_clipped_mean': summary['alpha_clipped_mean'][idx],
            'lambda_max_mean': summary['lambda_max_mean'][idx],
            'lambda_median_mean': summary['lambda_median_mean'][idx],
            'lambda_min_mean': summary['lambda_min_mean'][idx],
            'outlier_ratio_mean': summary['outlier_ratio_mean'][idx],
            'clip_effect_mean': summary['clip_effect_mean'][idx],
            'loss': summary['losses'][idx],
        })

trajectory_df = pd.DataFrame(trajectory_rows)
display(trajectory_df)


## Final spectrum view

The figure below compares the **final per-layer spectra** directly. For each layer, the initial spectrum is shown in gray for reference, and the final SGD and Muon spectra are overlaid on log-log axes.

This is still a per-layer toy diagnostic, but it is the most direct visual check of whether the optimizer difference looks like a bulk-spectrum difference, a top-eigenvalue difference, or some mixture of both.


In [ ]:
config = experiment['config']
dim = int(config['dim'])
num_layers = int(config['num_layers'])
ranks = np.arange(1, dim + 1)

sgd_result = experiment['optimizers']['SGD']
muon_result = experiment['optimizers']['Muon']

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.ravel()

for layer_idx in range(num_layers):
    ax = axes[layer_idx]
    ax.loglog(ranks, sgd_result['initial_eigenvalues'][layer_idx], color='gray', linestyle=':', linewidth=2.0, label='Initial' if layer_idx == 0 else None)
    ax.loglog(ranks, sgd_result['final_eigenvalues'][layer_idx], 'o-', color='tab:blue', markersize=3, linewidth=1.8, label='SGD' if layer_idx == 0 else None)
    ax.loglog(ranks, muon_result['final_eigenvalues'][layer_idx], 's--', color='tab:red', markersize=3, linewidth=1.8, label='Muon' if layer_idx == 0 else None)
    ax.set_title(f'Layer {layer_idx}')
    ax.set_xlabel('Rank')
    ax.set_ylabel('Eigenvalue of $W^T W$')
    ax.grid(True, which='both', alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=True)
fig.suptitle('Final per-layer spectra (with initial spectrum as reference)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()
plt.close(fig)

final_layer_rows = []
final_measure_sgd = -1
final_measure_muon = -1
for layer_idx in range(num_layers):
    final_layer_rows.append({
        'layer': layer_idx,
        'sgd_alpha_final': sgd_result['alpha'][final_measure_sgd, layer_idx],
        'muon_alpha_final': muon_result['alpha'][final_measure_muon, layer_idx],
        'sgd_alpha_clipped_final': sgd_result['alpha_clipped'][final_measure_sgd, layer_idx],
        'muon_alpha_clipped_final': muon_result['alpha_clipped'][final_measure_muon, layer_idx],
        'sgd_lambda_max_final': sgd_result['lambda_max'][final_measure_sgd, layer_idx],
        'muon_lambda_max_final': muon_result['lambda_max'][final_measure_muon, layer_idx],
        'sgd_outlier_ratio_final': sgd_result['outlier_ratio'][final_measure_sgd, layer_idx],
        'muon_outlier_ratio_final': muon_result['outlier_ratio'][final_measure_muon, layer_idx],
    })

final_layer_df = pd.DataFrame(final_layer_rows)
display(final_layer_df)


### Final-spectrum interpretation

A careful reading of the final spectra should distinguish two claims:

1. **Supported here**: Muon ends with a flatter layer-mean rank-decay proxy and slightly slower `lambda_max` growth than SGD.
2. **Not supported here**: the stronger story that clipping away the top eigenvalue matters *more* for SGD than for Muon.

That distinction matters because the current run suggests a real optimizer difference, but not exactly the originally overclaimed one.


## Deterministic checks T1--T4

The pair still uses the same four legacy checks, but they are presented here as **deterministic single-seed checks**, not as formal statistical tests.


In [ ]:
checks_df = pd.DataFrame([
    {
        'check': test['label'],
        'claim': test['claim'],
        'metric': test['metric_name'],
        'SGD_value': test['sgd_value'],
        'Muon_value': test['muon_value'],
        'supported': test['passed'],
        'result': test['outcome_text'],
        'note': test['note'],
    }
    for test in experiment['checks']['tests_in_order']
])
display(checks_df)

print(f"Legacy pass rule: {experiment['checks']['legacy_pass_rule']}")
print(f"Checks passed: {experiment['checks']['tests_passed']}/{experiment['checks']['tests_total']}")
print(f"Overall verdict: {experiment['summary']['overall']}")


## Calibrated conclusion

The notebook conclusion should match the returned results rather than a hard-coded story. In particular, it should separate the supported T1/T2/T3 narrative from the unsupported T4 clipping-effect narrative.


In [ ]:
print(textwrap.fill(experiment['summary']['calibrated_conclusion'], width=110))

print()
print('Short caveat list:')
caveats = [
    'single seed only',
    'per-layer spectrum proxy only',
    'alpha is a rank-decay proxy rather than a full WeightWatcher fit',
    'SGD learning-rate choice is heuristic rather than a formal fairness guarantee',
    'no product-spectrum or generalization measurements are included here',
]
for idx, item in enumerate(caveats, start=1):
    print(f'  {idx}. {item}')
